# DS105 Final Project
## Time-Based Browsing Pattern Analyzer with Deep Learning & RAM Correlation

This notebook walks through the full pipeline step by step with explanations.


## Step 0: Setup

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
print('Setup complete!')

## Step 1: Generate Sample Data

In [ ]:
os.chdir('..')  # Go to project root
from collect.generate_sample import generate_browsing_history, generate_ram_log, generate_domain_category_map

browsing = generate_browsing_history(days=5)
ram = generate_ram_log(days=5)
domain_map = generate_domain_category_map()

print(browsing.head())

## Step 2: Preprocess

In [ ]:
from prep.preprocess import preprocess

df = preprocess(
    input_path='data/browsing_history.csv',
    output_path='data/browsing_history.csv',
    map_path='data/domain_category_map.csv'
)

print('Category distribution:')
print(df['category'].value_counts())

## Step 3: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly activity
hourly = df.groupby('hour')['domain'].count()
axes[0].bar(hourly.index, hourly.values, color='steelblue')
axes[0].set_title('Visits by Hour')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Visits')

# Category pie
cat_counts = df['category'].value_counts()
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%')
axes[1].set_title('Category Distribution')

plt.tight_layout()
plt.show()

## Step 4: Sessionization

In [ ]:
from prep.sessionize import sessionize

sessions = sessionize(
    input_path='data/browsing_history.csv',
    output_path='data/sessions.csv',
    gap_minutes=15
)

print(f'Total sessions: {len(sessions)}')
print(sessions[['session_id','duration_min','top_category','social_ratio']].head(10))

## Step 5: RAM Correlation

In [ ]:
from analytics.ram_correlation import merge_ram

merged, ram_by_cat, ram_by_hour = merge_ram(
    browsing_path='data/browsing_history.csv',
    ram_path='data/ram_log.csv',
    output_path='data/merged_browsing_ram.csv'
)

print('RAM by category:')
print(ram_by_cat)

## Step 6: Clustering

In [ ]:
from models.clustering import run_clustering

sessions_clust, score, profiles = run_clustering(
    sessions_path='data/sessions.csv',
    output_path='data/sessions_clustered.csv',
    n_clusters=4
)

print(f'Silhouette Score: {score:.4f}')
print(sessions_clust['cluster_label'].value_counts())

## Step 7: Autoencoder Anomaly Detection

In [ ]:
from models.autoencoder import run_autoencoder

sessions_final, errors, threshold = run_autoencoder(
    sessions_path='data/sessions_clustered.csv',
    output_path='data/sessions_clustered.csv',
    epochs=30,
    anomaly_percentile=95
)

print(f'Anomalies detected: {sessions_final["is_anomaly"].sum()}')

## Step 8: Recommendations

In [ ]:
from analytics.recommendations import generate_recommendations

recs = generate_recommendations(
    sessions_path='data/sessions_clustered.csv',
    ram_by_cat_path='data/ram_by_category.csv'
)

## Step 9: Visualizations

In [ ]:
from analytics.visualizations import run_all_visualizations
run_all_visualizations()
print('All figures saved to reports/figures/')

## Done! Generate Final Report

In [ ]:
from analytics.report_generator import generate_report
generate_report()
print('Report saved to reports/final_report.md')